In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

PATH = os.path.dirname(os.path.abspath(""))
DATA_PATH = os.path.join(os.path.dirname(PATH), "data")

sys.path.append(PATH)

import src.dashboard_helpers.dashboard_utils as ut
import src.dashboard_plots.dashboard_plots_comparison as dbp

In [ ]:
data = pd.read_csv(DATA_PATH + '/full_data_2026-01-09.csv')
print(data.shape)
data.head()

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp['played_at'] = pd.to_datetime(tmp['played_at']).dt.strftime('%Y-%m-%d %H:%M:%S')
tmp

In [ ]:
pd.to_datetime(tmp['played_at'][0])

## Data Cleaning

- simplify artist_genres column
    - only use first 
    - only use most prevalent
- fill missing values
    - if a song (same song title + same combined_artist_names) occures more than once but one (or more) are lacking audio features
- convert played_at to Datetime 

#### Fill missing feature values

If a song (same song title + same combined_artist_names) occurs more than once but one (or more) are lacking audio features, then fill the missing values accordingly.

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

In [ ]:
# Convert played_at to datetime
data['played_at'] = pd.to_datetime(data['played_at'],format='ISO8601').dt.strftime('%Y-%m-%d %H:%M:%S')
data = ut.fill_missing_features(data=data)
data = ut.assign_main_genre(data=data)
print(data.shape)
data = ut.assign_simplified_main_genre(data=data)
print(data.shape)

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

## Data Enrichment

- track_duration_ms to track_duration_sec
- track_duration_ms to track_duration_min_sec (mm:ss)
- seconds_till_next_song
    - sort all data points by played_at (in ascending order)
    - calculate delta between current and next song in seconds
- song_fully_played_flag
    - IF track_duration_sec 0.95 <= seconds_till_next_song THEN 1 ELSE 0
        - we multiply with 0.95 here account for potential inaccuracies of played_at timestamp
        - besides, if the user listened to 90% of the song, we can consider it fully played

In [ ]:
data = ut.convert_track_duration(data=data)
data = ut.add_fully_played_label(data=data, fully_played_threshold=0.95)
data

In [ ]:
import requests

url = "https://api.reccobeats.com/v1/track/recommendation?size=5&seeds=6WrusttQhWlojn3WKlPTaY"

payload = {}
headers = {
  'Accept': 'application/json'
}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)

In [ ]:
data.loc[data['track_name']=='Tempest',:].index[0]

In [ ]:
np.unique(data['main_genre'].apply(str))

In [ ]:
data.loc[data['track_name']=='Pompidou - Chanknous Remix', :]

In [ ]:
data['artist_genres'].isnull().sum()

## Visualizations

In [ ]:
plot_fig = dbp.song_comparison_radar_chart(data=data, idx_main=0, idx_ref=7)
plot_fig.show()

__Track + Album Information:__
- track_name
- album_name
- track_duration_min_sec
- track_popularity
- album_image_url

__Artist Information:__
- combined_artist_name
- main_genre
- artist_followers
- artist_popularity
- artist_image_url

__Additional Track Stats:__
- key + mode
- loudness
- tempo (bpm) 

In [ ]:
track_idx = 7
track_type = 'ref' # 'main'

plot_fig = dbp.song_comparison_audio_feature_bar_chart(data=data, track_idx=track_idx, track_type=track_type)
plot_fig.show()

In [ ]:
feature_name = 'loudness'  # 'loudness' or 'tempo'

plot_fig = dbp.song_comparison_polar_chart(data=data, feature_name=feature_name, idx_main=0, idx_ref=7)
plot_fig.show()

In [ ]:
360 / (60 + data['loudness'].max())

In [ ]:
# Define feature ('loudness' or 'tempo')
feature = 'tempo' # 'loudness'

# Set feature label and range
if feature == 'loudness':
    feature_label = '<br>Loudness (dB)'
    feature_range = [np.round(data['loudness'].min() / 10, 0) * 10, 0]
else:
    feature_label = '<br>Tempo (BPM)'
    feature_range = [0, np.round(data['tempo'].max() / 10, 0) * 10]

# Get track names
track_main = data.loc[0, 'track_name']
track_ref = data.loc[7, 'track_name']

# Get loudness and tempo values
feature_main = data.loc[0, feature]
feature_ref = data.loc[7, feature]

# Create figure
fig = go.Figure()

# Add bars for main track
fig.add_trace(go.Bar(
    name=track_main,
    x=[feature_label],
    y=[feature_main],
    #marker=dict(color='rgb(57, 255, 0)'),  # Neon green 'rgba(30, 215, 96, 0.9)'
    marker=dict(color='rgb(30, 215, 96)'),  # darker green
    text=[f'{feature_main:.1f}'],
    textposition='outside',
    hovertemplate=f'<b>Main Track: </b>{track_main}<br>{feature_label}: {feature_main:.1f}<extra></extra>',
    width=0.15
))

# Add bars for reference track
fig.add_trace(go.Bar(
    name=track_ref,
    x=[feature_label],
    y=[feature_ref],
    marker=dict(color='rgb(0, 191, 255)'),  # Neon blue
    text=[f'{feature_ref:.1f}'],
    textposition='outside',
    hovertemplate=f'<b>Reference Track: </b>{track_ref}<br>{feature_label}: {feature_ref:.1f}<extra></extra>',
    width=0.15
))

# Update layout
fig.update_layout(
    barmode='group',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white', size=14),
    xaxis=dict(gridcolor='rgba(255, 255, 255, 0.2)'),
    yaxis=dict(
        gridcolor='rgba(255, 255, 255, 0.2)',
        range=feature_range
    ),
    showlegend=False
)

fig.show()